# L33 · DPO：最简单的高效对齐

**学习目标**
- 理解 DPO（Direct Preference Optimization）为什么「不用单独训练奖励模型」
- 理解「策略模型 vs 参考模型」的约束（别跑偏）
- 用 numpy 跑一个 DPO，看偏好差距拉大且不崩

**前置依赖**：L31（SFT）、L32（奖励模型）  
**预计时长**：55 分钟  
**技术栈**：`numpy`、`matplotlib`（离线可运行）

---

## 概念讲解：DPO = 把「训评委」和「训选手」合成一步

传统 RLHF（L35）要：① 先训奖励模型 → ② 再用它强化策略。又贵又绕。
**DPO** 一句话点破：偏好数据本身就编码了「好>坏」，可以直接拿它当损失，
**一步** 把策略模型调向「偏好回答」、调离「不偏好回答」。

关键约束：不能调太狠，否则模型「忘本」。所以要拿一个**参考模型**（训练前的快照）当缰绳。

## 第一步：准备偏好对 + 参考模型快照

In [ ]:
import numpy as np
np.random.seed(2)
good = np.random.randn(60, 3) + 1.0
bad = np.random.randn(60, 3) - 1.0
pairs = list(zip(good, bad))

ref_W = np.random.randn(3) * 0.1       # 参考模型（冻结，当缰绳）
pol_W = ref_W.copy()                    # 策略模型（我们要训练的）
beta = 0.5                             # 缰绳强度
print("DPO 数据就绪：", len(pairs), "对偏好；参考模型已冻结")

## 第二步：DPO 损失与优化

In [ ]:
def sigmoid(x): return 1 / (1 + np.exp(-x))
lr, losses = 0.05, []
for step in range(300):
    grad = np.zeros(3); total = 0
    for g, b in pairs:
        # 策略模型对好/坏的优势
        adv_g = pol_W @ g - ref_W @ g
        adv_b = pol_W @ b - ref_W @ b
        # DPO 损失：让 (好优势 - 坏优势) 尽量大
        logits = beta * (adv_g - adv_b)
        p = sigmoid(logits)
        total += -np.log(p + 1e-8)
        grad += beta * (p - 1) * (g - b)
    losses.append(total / len(pairs))
    pol_W -= lr * grad / len(pairs)
print(f"DPO 完成：损失 {losses[0]:.3f} → {losses[-1]:.3f}")
print(f"策略模型最终权重：{np.round(pol_W, 2)}（参考模型：{np.round(ref_W, 2)}）")

# 🎯 AHA 顿悟单元格：DPO 一步对齐 · 偏好差距被拉开

运行下面代码。你会看到：DPO 训练后，**策略模型对好回答的打分显著高于坏回答**，
同时因为「参考模型缰绳」的存在，它没有偏离太远（看权重变化幅度）。
一张图对比「训练前后偏好差距」+ 损失曲线。

> 你刚实现的，就是 2023 年引爆业界的 DPO 算法的最小内核。它让对齐从「训两个模型」变成「训一个」，
> 今天绝大多数开源对齐（Zephyr、Tulu）都靠它。

In [ ]:
# ===== 运行我！看 DPO 对齐效果 =====
import matplotlib.pyplot as plt
g_s = [pol_W @ g for g, _ in pairs]
b_s = [pol_W @ b for _, b in pairs]
ref_g = [ref_W @ g for g, _ in pairs]
ref_b = [ref_W @ b for _, b in pairs]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, color="#9467bd"); ax[0].set_title("DPO 训练损失")
ax[0].set_xlabel("步"); ax[0].set_ylabel("损失")
x = np.arange(2)
ax[1].bar(x-0.2, [np.mean(ref_g), np.mean(ref_b)], 0.4, label="参考模型", color="#999")
ax[1].bar(x+0.2, [np.mean(g_s), np.mean(b_s)], 0.4, label="DPO后策略", color=["#2ca02c","#ff7f0e"])
ax[1].set_xticks(x); ax[1].set_xticklabels(["好回答","坏回答"])
ax[1].set_title("对齐后：好回答分↑ 坏回答分↓"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"  🎯 策略模型偏好差距：{np.mean(g_s)-np.mean(b_s):.2f}")
print(f"  🔒 与参考模型偏离度：{np.linalg.norm(pol_W-ref_W):.2f}（受 beta 约束，未崩）")
print("  ✨ 你用 DPO 一步完成了对齐——这是现代开源 LLM 的标配技术！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：DPO 为何无需显式 RM（直接从偏好推导策略梯度等价形式）；β 作为 KL 约束强度的角色。  
**易错点**：参考模型必须冻结（本方案 ref_W 不参与更新）；数值稳定（log 加 1e-8）。  
**AHA 机制**：对齐前后对比图 + 偏离度，强「一步对齐且不崩」实感。  
**衔接**：L34 PPO（DPO 的强化学习前身，更通用但更复杂）；L35 RLHF 串联；L39 后训练管线。  
**依赖**：`pip install numpy matplotlib`。  
**真 LLM 路径**：真实 DPO 用 `transformers` + `DPOTrainer`（HuggingFace TRL），在 SFT 模型上做；本演示是线性投影版。

# 📚 作业 / 下一步

1. 把 `beta` 改成 0.1，看策略模型是否过度偏离参考（对比偏离度）。
2. 把 `beta` 改成 5.0，看对齐是否变弱（约束太强）。
3. 下一课 **L34 PPO 基础：强化学习直觉** —— 理解 DPO 背后那套更通用的强化学习引擎。